# LLM Pipeline Profiler — Kaggle Notebook Example

This notebook demonstrates how to instrument a real HuggingFace LLM inference pipeline using `llm-profiler`.

It installs the profiler package, downloads the `distilgpt2` model, runs the inference generation loop stage-by-stage (loading, tokenization, autoregressive token generation, and post-processing decoding), and uploads the time-series throughput, CPU/RAM/GPU memory stats, and execution trace to your local/ngrok dashboard.

In [ ]:
# 1. Install the profiler library directly from GitHub
# This automatically builds the high-frequency C++ sampler extension during installation.
!pip install git+https://github.com/harshgupta-23/llm-pipeline-profiler.git#subdirectory=profiler-lib

In [ ]:
# 2. Setup the imports and configure the Tracer
import torch
import time
from transformers import AutoModelForCausalLM, AutoTokenizer
from llm_profiler import Tracer

# Auto-detect device (GPU/CUDA if available on Kaggle, else CPU)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running LLM generation profile on device: {device}")

# Initialize Tracer
# Replace 'dashboard_url' with your actual ngrok forwarding URL (e.g. "https://xxxx.ngrok-free.dev")
# Or set dashboard_url=None to run in offline mode (saves results to /kaggle/working/run.json)
tracer = Tracer(
    run_name="kaggle-distilgpt2-run",
    model_name="distilgpt2",
    dashboard_url="https://stimuli-detention-video.ngrok-free.dev"
)

In [ ]:
# 3. Stage 1: Load the Model and Tokenizer
with tracer.stage("model_load"):
    print("Loading model and tokenizer...")
    model = AutoModelForCausalLM.from_pretrained("distilgpt2").to(device)
    tokenizer = AutoTokenizer.from_pretrained("distilgpt2")

In [ ]:
# 4. Stage 2: Tokenization
with tracer.stage("tokenize"):
    prompt = "Deep learning and artificial intelligence are revolutionizing"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

In [ ]:
# 5. Stage 3: Autoregressive Token Generation (with C++ sampling & PyTorch trace)
with tracer.stage("generate", profile_torch=True):
    print("Starting generation loop...")
    input_ids = inputs["input_ids"]
    attention_mask = inputs["attention_mask"]
    
    start_time = time.perf_counter()
    tokens_generated = 0
    max_new_tokens = 40

    for i in range(max_new_tokens):
        with torch.no_grad():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            next_token = torch.argmax(outputs.logits[:, -1, :], dim=-1, keepdim=True)
            
            # Append generated token to inputs
            input_ids = torch.cat([input_ids, next_token], dim=-1)
            attention_mask = torch.cat([
                attention_mask,
                torch.ones((1, 1), device=device, dtype=torch.long)
            ], dim=-1)
            
        tokens_generated += 1
        elapsed = time.perf_counter() - start_time
        
        # Calculate tokens per second (throughput) and log it
        tps = tokens_generated / elapsed if elapsed > 0 else 0.0
        tracer.log_metric("tps", tps)
        
        # Slight delay to mimic processing latency and allow metrics sampling
        time.sleep(0.02)

In [ ]:
# 6. Stage 4: Post-processing (Decoding)
with tracer.stage("postprocess"):
    output_text = tokenizer.decode(input_ids[0], skip_special_tokens=True)

print("\n--- Generated Output ---")
print(output_text)
print("------------------------\n")

In [ ]:
# 7. Export pipeline profiling data to dashboard
print("Exporting pipeline profiling data to dashboard...")
tracer.export()
print("Success! View the dashboard details page for this run to see all charts and flamegraphs.")